In [ ]:
# ══════════════════════════════════════════════════════════════════
# M5 Quiz Setup — self-contained. No git clone needed.
# Before running: Runtime → Change runtime type → T4 GPU
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash kailash-ml kailash-kaizen polars plotly python-dotenv nest-asyncio torchvision

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. Run the next cell to load quiz helpers.')


In [ ]:
from __future__ import annotations


# ── kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model

# ── diagnostics.py ──
"""DL Diagnostics Toolkit — clinical instruments for deep-network training.

Wraps PyTorch forward/backward hooks into a student-friendly API that
surfaces the four failure modes professionals must recognise:

    1. Stethoscope  — loss-curve shape (under/over-fit, instability)
    2. Blood Test   — gradient flow per layer (vanishing / exploding)
    3. X-Ray        — activation statistics & dead neurons
    4. Prescription — automated diagnosis with actionable suggestions

Typical use inside an exercise::


    with DLDiagnostics(model) as diag:
        diag.track_gradients()
        diag.track_activations()
        diag.track_dead_neurons()
        for epoch in range(epochs):
            for batch in dataloader:
                loss = train_step(model, batch)
                diag.record_batch(loss=loss.item(),
                                  lr=optimizer.param_groups[0]["lr"])
            diag.record_epoch(val_loss=evaluate(model, val_loader))
        diag.plot_training_dashboard().show()
        diag.report()

All DataFrames are Polars. All plots are Plotly. No matplotlib, no pandas.
"""

import logging
import math
from collections.abc import Callable
from dataclasses import dataclass, field
from typing import Any

import numpy as np
import plotly.graph_objects as go
import polars as pl
import torch
import torch.nn as nn
from plotly.subplots import make_subplots
from torch.utils.data import DataLoader


logger = logging.getLogger(__name__)

# Module names whose outputs are "dead-neuron sensitive" — we track fraction
# of zero outputs for these specifically.
_DEAD_NEURON_SENSITIVE: tuple[type[nn.Module], ...] = (
    nn.ReLU,
    nn.LeakyReLU,
    nn.GELU,
    nn.ELU,
    nn.SiLU,
)

# Module names whose outputs we monitor for activation statistics.
_ACTIVATION_MONITORED: tuple[type[nn.Module], ...] = (
    nn.Linear,
    nn.Conv1d,
    nn.Conv2d,
    nn.Conv3d,
    nn.ReLU,
    nn.LeakyReLU,
    nn.GELU,
    nn.ELU,
    nn.SiLU,
    nn.Tanh,
    nn.Sigmoid,
    nn.BatchNorm1d,
    nn.BatchNorm2d,
    nn.LayerNorm,
)


@dataclass
class _HookHandles:
    """Container for registered hook handles so we can detach cleanly."""

    gradient: list[torch.utils.hooks.RemovableHandle] = field(default_factory=list)
    activation: list[torch.utils.hooks.RemovableHandle] = field(default_factory=list)
    dead_neuron: list[torch.utils.hooks.RemovableHandle] = field(default_factory=list)
    grad_cam: list[torch.utils.hooks.RemovableHandle] = field(default_factory=list)

    def all(self) -> list[torch.utils.hooks.RemovableHandle]:
        return self.gradient + self.activation + self.dead_neuron + self.grad_cam


class DLDiagnostics:
    """Clinical instrumentation for PyTorch training loops.

    Collects per-batch time series of gradient norms, activation statistics,
    dead-neuron fractions, and losses; exposes Plotly visualisations and an
    automated report that surfaces overfitting, vanishing gradients, and
    pathological dead-ReLU layers.

    Args:
        model: The ``nn.Module`` to instrument. The model is NOT modified;
            only forward/backward hooks are attached.
        dead_neuron_threshold: Fraction of zero outputs above which a layer
            is flagged as "dead" in :meth:`report`. Defaults to ``0.5``.
        window: Number of recent batches used for dead-neuron statistics.
            Defaults to ``64``.

    Example:
        >>> model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 1))
        >>> with DLDiagnostics(model) as diag:
        ...     diag.track_gradients()
        ...     diag.track_activations()
        ...     # ... training loop ...
    """

    def __init__(
        self,
        model: nn.Module,
        *,
        dead_neuron_threshold: float = 0.5,
        window: int = 64,
    ) -> None:
        if not isinstance(model, nn.Module):
            raise TypeError(
                f"DLDiagnostics requires an nn.Module; got {type(model).__name__}"
            )
        if not 0.0 < dead_neuron_threshold < 1.0:
            raise ValueError("dead_neuron_threshold must be in (0, 1)")
        if window < 1:
            raise ValueError("window must be >= 1")

        self.model = model
        self.device = get_device()
        self.dead_neuron_threshold = dead_neuron_threshold
        self.window = window

        # Time series storage — lists of dicts, converted to Polars on demand.
        self._grad_log: list[dict[str, Any]] = []
        self._act_log: list[dict[str, Any]] = []
        self._dead_log: list[dict[str, Any]] = []
        self._batch_log: list[dict[str, Any]] = []
        self._epoch_log: list[dict[str, Any]] = []

        # Running per-layer firing masks for dead-neuron detection.
        # Key: layer name -> tensor of firing counts per neuron (1D).
        self._firing_counts: dict[str, torch.Tensor] = {}
        self._firing_samples: dict[str, int] = {}

        # Counters bound to hook closures so they share scope.
        self._batch_idx = 0
        self._epoch_idx = 0

        self._handles = _HookHandles()
        self._tracking = {"gradients": False, "activations": False, "dead": False}

        # Grad-CAM capture buffers (populated on demand).
        self._gradcam_activation: torch.Tensor | None = None
        self._gradcam_gradient: torch.Tensor | None = None

        logger.info(
            "dldiagnostics.init",
            extra={
                "model_class": type(model).__name__,
                "device": str(self.device),
                "window": window,
            },
        )

    # ── Context-manager support ────────────────────────────────────────────

    def __enter__(self) -> DLDiagnostics:
        return self

    def __exit__(self, exc_type, exc, tb) -> None:  # noqa: D401
        self.detach()

    def __del__(self) -> None:  # pragma: no cover - best-effort cleanup
        try:
            self.detach()
        except Exception:
            # Finalizers MUST NOT raise. Silent cleanup is the documented
            # contract for __del__ in rules/patterns.md.
            pass

    # ── Hook registration ──────────────────────────────────────────────────

    def track_gradients(self) -> DLDiagnostics:
        """Register backward hooks on every trainable parameter.

        Records the L2 norm of each parameter's gradient at every backward
        pass, keyed by parameter name.

        Returns:
            ``self`` for chaining.
        """
        if self._tracking["gradients"]:
            return self
        for name, param in self.model.named_parameters():
            if not param.requires_grad:
                continue
            handle = param.register_hook(self._make_grad_hook(name))
            self._handles.gradient.append(handle)
        self._tracking["gradients"] = True
        logger.info(
            "dldiagnostics.track_gradients",
            extra={"hooks_registered": len(self._handles.gradient)},
        )
        return self

    def track_activations(self) -> DLDiagnostics:
        """Register forward hooks on monitored submodules.

        Records mean/std/min/max/dead-fraction of activations per layer
        at every forward pass.

        Returns:
            ``self`` for chaining.
        """
        if self._tracking["activations"]:
            return self
        for name, module in self.model.named_modules():
            if name == "" or not isinstance(module, _ACTIVATION_MONITORED):
                continue
            handle = module.register_forward_hook(self._make_act_hook(name))
            self._handles.activation.append(handle)
        self._tracking["activations"] = True
        logger.info(
            "dldiagnostics.track_activations",
            extra={"hooks_registered": len(self._handles.activation)},
        )
        return self

    def track_dead_neurons(self) -> DLDiagnostics:
        """Register forward hooks on ReLU-family layers to track dead neurons.

        A "neuron" here is a channel (Conv) or output unit (Linear). The
        rolling fraction of batches where that neuron output zero is tracked.

        Returns:
            ``self`` for chaining.
        """
        if self._tracking["dead"]:
            return self
        for name, module in self.model.named_modules():
            if name == "" or not isinstance(module, _DEAD_NEURON_SENSITIVE):
                continue
            handle = module.register_forward_hook(self._make_dead_hook(name))
            self._handles.dead_neuron.append(handle)
        self._tracking["dead"] = True
        logger.info(
            "dldiagnostics.track_dead_neurons",
            extra={"hooks_registered": len(self._handles.dead_neuron)},
        )
        return self

    def detach(self) -> None:
        """Remove ALL registered hooks and release references.

        Safe to call multiple times. Invoked automatically on context exit.
        """
        for handle in self._handles.all():
            try:
                handle.remove()
            except Exception:
                # Hook removal failures are benign (module may already be
                # gone). See rules/zero-tolerance.md Rule 3 carve-out for
                # cleanup paths.
                pass
        self._handles = _HookHandles()
        self._tracking = {k: False for k in self._tracking}
        self._gradcam_activation = None
        self._gradcam_gradient = None

    # ── Recording ─────────────────────────────────────────────────────────

    def record_batch(self, *, loss: float, lr: float | None = None) -> None:
        """Record per-batch scalar training signals.

        Args:
            loss: Training loss for the batch (post-backward).
            lr: Current learning rate (optional; read from optimizer).
        """
        if not math.isfinite(loss):
            logger.warning(
                "dldiagnostics.record_batch.nonfinite_loss",
                extra={"loss": str(loss), "batch": self._batch_idx},
            )
        self._batch_log.append(
            {
                "batch": self._batch_idx,
                "epoch": self._epoch_idx,
                "loss": float(loss),
                "lr": float(lr) if lr is not None else float("nan"),
            }
        )
        self._batch_idx += 1

    def record_epoch(
        self,
        *,
        val_loss: float | None = None,
        train_loss: float | None = None,
        **extra: float,
    ) -> None:
        """Record per-epoch summary metrics.

        Args:
            val_loss: Validation loss at epoch end.
            train_loss: Mean training loss for the epoch. If ``None``, it is
                computed from the batches in this epoch.
            **extra: Any additional scalar metrics to persist.
        """
        if train_loss is None:
            epoch_batches = [
                b for b in self._batch_log if b["epoch"] == self._epoch_idx
            ]
            if epoch_batches:
                train_loss = float(np.mean([b["loss"] for b in epoch_batches]))
        entry = {
            "epoch": self._epoch_idx,
            "train_loss": train_loss if train_loss is not None else float("nan"),
            "val_loss": val_loss if val_loss is not None else float("nan"),
            **{k: float(v) for k, v in extra.items()},
        }
        self._epoch_log.append(entry)
        self._epoch_idx += 1

    # ── Public DataFrame accessors ────────────────────────────────────────

    def gradients_df(self) -> pl.DataFrame:
        """Polars DataFrame of per-layer gradient norms by batch."""
        if not self._grad_log:
            return pl.DataFrame(
                schema={
                    "batch": pl.Int64,
                    "layer": pl.Utf8,
                    "grad_norm": pl.Float64,
                    "grad_rms": pl.Float64,
                    "update_ratio": pl.Float64,
                }
            )
        return pl.DataFrame(self._grad_log)

    def activations_df(self) -> pl.DataFrame:
        """Polars DataFrame of per-layer activation statistics by batch."""
        if not self._act_log:
            return pl.DataFrame(
                schema={
                    "batch": pl.Int64,
                    "layer": pl.Utf8,
                    "act_kind": pl.Utf8,
                    "mean": pl.Float64,
                    "std": pl.Float64,
                    "min": pl.Float64,
                    "max": pl.Float64,
                    "dead_fraction": pl.Float64,
                    "inactivity_fraction": pl.Float64,
                }
            )
        return pl.DataFrame(self._act_log)

    def dead_neurons_df(self) -> pl.DataFrame:
        """Polars DataFrame of current per-layer dead-neuron fractions."""
        rows: list[dict[str, Any]] = []
        for name, counts in self._firing_counts.items():
            n_samples = max(self._firing_samples.get(name, 1), 1)
            dead_mask = (counts / n_samples) < 1e-6
            rows.append(
                {
                    "layer": name,
                    "n_neurons": int(counts.numel()),
                    "n_dead": int(dead_mask.sum().item()),
                    "dead_fraction": float(dead_mask.float().mean().item()),
                }
            )
        if not rows:
            return pl.DataFrame(
                schema={
                    "layer": pl.Utf8,
                    "n_neurons": pl.Int64,
                    "n_dead": pl.Int64,
                    "dead_fraction": pl.Float64,
                }
            )
        return pl.DataFrame(rows)

    def batches_df(self) -> pl.DataFrame:
        """Polars DataFrame of per-batch scalars (loss, lr)."""
        if not self._batch_log:
            return pl.DataFrame(
                schema={
                    "batch": pl.Int64,
                    "epoch": pl.Int64,
                    "loss": pl.Float64,
                    "lr": pl.Float64,
                }
            )
        return pl.DataFrame(self._batch_log)

    def epochs_df(self) -> pl.DataFrame:
        """Polars DataFrame of per-epoch summary metrics."""
        if not self._epoch_log:
            return pl.DataFrame(
                schema={
                    "epoch": pl.Int64,
                    "train_loss": pl.Float64,
                    "val_loss": pl.Float64,
                }
            )
        return pl.DataFrame(self._epoch_log)

    # ── Plots ─────────────────────────────────────────────────────────────

    def plot_loss_curves(self) -> go.Figure:
        """Loss stethoscope: train vs val curves with overfitting callout.

        Returns:
            A Plotly Figure with two traces (train / val) and annotations
            for detected overfitting epoch (if any).
        """
        epochs = self.epochs_df()
        batches = self.batches_df()
        fig = go.Figure()
        if batches.height:
            fig.add_trace(
                go.Scatter(
                    x=batches["batch"].to_list(),
                    y=batches["loss"].to_list(),
                    mode="lines",
                    name="train (per batch)",
                    line=dict(color="lightblue", width=1),
                    opacity=0.5,
                )
            )
        if epochs.height:
            fig.add_trace(
                go.Scatter(
                    x=epochs["epoch"].to_list(),
                    y=epochs["train_loss"].to_list(),
                    mode="lines+markers",
                    name="train (epoch mean)",
                    line=dict(color="steelblue", width=2),
                )
            )
            if epochs["val_loss"].is_not_null().any():
                fig.add_trace(
                    go.Scatter(
                        x=epochs["epoch"].to_list(),
                        y=epochs["val_loss"].to_list(),
                        mode="lines+markers",
                        name="val",
                        line=dict(color="firebrick", width=2),
                    )
                )
        overfit_epoch = self._detect_overfit_epoch()
        if overfit_epoch is not None:
            fig.add_vline(
                x=overfit_epoch,
                line=dict(color="orange", dash="dash"),
                annotation_text=f"overfitting suspected @ epoch {overfit_epoch}",
                annotation_position="top",
            )
        fig.update_layout(
            title="Loss Curves (Stethoscope)",
            xaxis_title="step",
            yaxis_title="loss",
            template="plotly_white",
            hovermode="x unified",
        )
        return fig

    def plot_gradient_flow(self) -> go.Figure:
        """Blood test: per-layer gradient norm over time.

        One trace per tracked parameter. Layers whose gradient norm collapses
        toward zero are the vanishing-gradient culprits.
        """
        df = self.gradients_df()
        fig = go.Figure()
        if df.height == 0:
            fig.update_layout(
                title="Gradient Flow (Blood Test) — no data",
                template="plotly_white",
            )
            return fig
        for layer in df["layer"].unique().to_list():
            sub = df.filter(pl.col("layer") == layer)
            fig.add_trace(
                go.Scatter(
                    x=sub["batch"].to_list(),
                    y=sub["grad_norm"].to_list(),
                    mode="lines",
                    name=layer,
                    hovertemplate=f"{layer}<br>batch=%{{x}}<br>‖∇‖=%{{y:.3e}}",
                )
            )
        fig.update_layout(
            title="Gradient Flow by Layer (Blood Test)",
            xaxis_title="batch",
            yaxis_title="gradient L2 norm",
            yaxis_type="log",
            template="plotly_white",
        )
        return fig

    def plot_activation_stats(self) -> go.Figure:
        """X-Ray: activation mean ± std per layer over time."""
        df = self.activations_df()
        fig = go.Figure()
        if df.height == 0:
            fig.update_layout(
                title="Activation Statistics (X-Ray) — no data",
                template="plotly_white",
            )
            return fig
        for layer in df["layer"].unique().to_list():
            sub = df.filter(pl.col("layer") == layer)
            fig.add_trace(
                go.Scatter(
                    x=sub["batch"].to_list(),
                    y=sub["mean"].to_list(),
                    mode="lines",
                    name=f"{layer} — mean",
                    hovertemplate=(
                        f"{layer}<br>batch=%{{x}}<br>mean=%{{y:.3f}}<br>"
                        "std=%{customdata:.3f}"
                    ),
                    customdata=sub["std"].to_list(),
                )
            )
        fig.update_layout(
            title="Activation Mean per Layer (X-Ray)",
            xaxis_title="batch",
            yaxis_title="activation mean",
            template="plotly_white",
        )
        return fig

    def plot_dead_neurons(self) -> go.Figure:
        """X-Ray: fraction of dead neurons per ReLU-family layer."""
        df = self.dead_neurons_df()
        fig = go.Figure()
        if df.height == 0:
            fig.update_layout(
                title="Dead Neurons (X-Ray) — no ReLU-family layers tracked",
                template="plotly_white",
            )
            return fig
        colors = [
            "firebrick" if frac > self.dead_neuron_threshold else "steelblue"
            for frac in df["dead_fraction"].to_list()
        ]
        fig.add_trace(
            go.Bar(
                x=df["layer"].to_list(),
                y=df["dead_fraction"].to_list(),
                marker_color=colors,
                text=[
                    f"{int(n)}/{int(t)}" for n, t in zip(df["n_dead"], df["n_neurons"])
                ],
                textposition="outside",
            )
        )
        fig.add_hline(
            y=self.dead_neuron_threshold,
            line=dict(color="orange", dash="dash"),
            annotation_text=f"alert threshold ({self.dead_neuron_threshold:.0%})",
        )
        fig.update_layout(
            title="Dead Neuron Fraction by Layer (X-Ray)",
            xaxis_title="layer",
            yaxis_title="fraction dead",
            yaxis=dict(range=[0, 1]),
            template="plotly_white",
            showlegend=False,
        )
        return fig

    def plot_training_dashboard(self) -> go.Figure:
        """Vital signs: 2x2 dashboard (loss, grad flow, activations, LR)."""
        fig = make_subplots(
            rows=2,
            cols=2,
            subplot_titles=(
                "Loss (Stethoscope)",
                "Gradient Flow (Blood Test)",
                "Activation Mean (X-Ray)",
                "Learning Rate",
            ),
        )

        # (1,1) Loss
        epochs = self.epochs_df()
        batches = self.batches_df()
        if batches.height:
            fig.add_trace(
                go.Scatter(
                    x=batches["batch"].to_list(),
                    y=batches["loss"].to_list(),
                    mode="lines",
                    name="train loss",
                    line=dict(color="steelblue", width=1),
                ),
                row=1,
                col=1,
            )
        if epochs.height and epochs["val_loss"].is_not_null().any():
            # Place val loss at the last batch of each epoch for alignment.
            val_x = []
            for ep in epochs["epoch"].to_list():
                ep_batches = batches.filter(pl.col("epoch") == ep)
                val_x.append(
                    int(ep_batches["batch"].max()) if ep_batches.height else ep
                )
            fig.add_trace(
                go.Scatter(
                    x=val_x,
                    y=epochs["val_loss"].to_list(),
                    mode="lines+markers",
                    name="val loss",
                    line=dict(color="firebrick"),
                ),
                row=1,
                col=1,
            )

        # (1,2) Gradient flow
        gdf = self.gradients_df()
        if gdf.height:
            for layer in gdf["layer"].unique().to_list():
                sub = gdf.filter(pl.col("layer") == layer)
                fig.add_trace(
                    go.Scatter(
                        x=sub["batch"].to_list(),
                        y=sub["grad_norm"].to_list(),
                        mode="lines",
                        name=layer,
                        showlegend=False,
                    ),
                    row=1,
                    col=2,
                )
            fig.update_yaxes(type="log", row=1, col=2)

        # (2,1) Activation mean
        adf = self.activations_df()
        if adf.height:
            for layer in adf["layer"].unique().to_list():
                sub = adf.filter(pl.col("layer") == layer)
                fig.add_trace(
                    go.Scatter(
                        x=sub["batch"].to_list(),
                        y=sub["mean"].to_list(),
                        mode="lines",
                        name=layer,
                        showlegend=False,
                    ),
                    row=2,
                    col=1,
                )

        # (2,2) LR
        if batches.height and batches["lr"].is_not_null().any():
            fig.add_trace(
                go.Scatter(
                    x=batches["batch"].to_list(),
                    y=batches["lr"].to_list(),
                    mode="lines",
                    name="lr",
                    line=dict(color="darkgreen"),
                    showlegend=False,
                ),
                row=2,
                col=2,
            )

        fig.update_layout(
            title="Training Dashboard (Vital Signs)",
            template="plotly_white",
            height=720,
        )
        return fig

    def plot_lr_vs_loss(self) -> go.Figure:
        """Plot LR vs loss (useful after an :meth:`lr_range_test`)."""
        df = self.batches_df()
        fig = go.Figure()
        if df.height == 0 or df["lr"].is_null().all():
            fig.update_layout(title="LR vs Loss — no data", template="plotly_white")
            return fig
        fig.add_trace(
            go.Scatter(
                x=df["lr"].to_list(),
                y=df["loss"].to_list(),
                mode="lines",
                line=dict(color="steelblue"),
            )
        )
        fig.update_layout(
            title="Learning Rate vs Loss",
            xaxis_title="learning rate",
            yaxis_title="loss",
            xaxis_type="log",
            template="plotly_white",
        )
        return fig

    def plot_weight_distributions(self) -> go.Figure:
        """Histogram of weight values, one trace per parameter tensor."""
        fig = go.Figure()
        for name, param in self.model.named_parameters():
            if not param.requires_grad or param.ndim == 0:
                continue
            values = param.detach().cpu().flatten().numpy()
            fig.add_trace(go.Histogram(x=values, name=name, opacity=0.6))
        fig.update_layout(
            title="Weight Distributions",
            xaxis_title="value",
            yaxis_title="count",
            barmode="overlay",
            template="plotly_white",
        )
        return fig

    def plot_gradient_norms(self) -> go.Figure:
        """Mean gradient norm per layer across the run (bar chart)."""
        df = self.gradients_df()
        fig = go.Figure()
        if df.height == 0:
            fig.update_layout(title="Gradient Norms — no data", template="plotly_white")
            return fig
        summary = df.group_by("layer").agg(
            pl.col("grad_norm").mean().alias("mean_grad")
        )
        summary = summary.sort("mean_grad")
        fig.add_trace(
            go.Bar(
                x=summary["layer"].to_list(),
                y=summary["mean_grad"].to_list(),
                marker_color="steelblue",
            )
        )
        fig.update_layout(
            title="Mean Gradient Norm per Layer",
            xaxis_title="layer",
            yaxis_title="mean ‖∇‖",
            yaxis_type="log",
            template="plotly_white",
        )
        return fig

    # ── Automated report ──────────────────────────────────────────────────

    def report(self) -> dict[str, Any]:
        """Print a human-readable diagnosis and return findings as a dict.

        Findings covered:
            * Gradient flow (vanishing / exploding / healthy)
            * Dead neurons (per-layer ReLU-family)
            * Loss trend (overfitting, underfitting, instability)

        Returns:
            A dict with keys ``gradient_flow``, ``dead_neurons``, ``loss_trend``
            each mapping to a dict with ``severity`` and ``message``.
        """
        findings: dict[str, Any] = {}

        # 1. Gradient flow — uses SCALE-INVARIANT per-element RMS (grad_rms)
        # and update_ratio (‖∇W‖/‖W‖), matching slide 5F thresholds.
        gdf = self.gradients_df()
        if gdf.height and "grad_rms" in gdf.columns:
            stats = gdf.group_by("layer").agg(
                pl.col("grad_rms").mean().alias("rms"),
                pl.col("update_ratio").mean().alias("ur"),
            )
            min_rms_raw = stats["rms"].min()
            max_rms_raw = stats["rms"].max()
            min_ur_raw = stats["ur"].min()
            max_ur_raw = stats["ur"].max()
            min_rms = (
                float(min_rms_raw) if isinstance(min_rms_raw, (int, float)) else None
            )
            max_rms = (
                float(max_rms_raw) if isinstance(max_rms_raw, (int, float)) else None
            )
            min_ur = float(min_ur_raw) if isinstance(min_ur_raw, (int, float)) else 0.0
            max_ur = float(max_ur_raw) if isinstance(max_ur_raw, (int, float)) else 0.0
            if min_rms is None or max_rms is None or min_rms == 0:
                severity = "UNKNOWN"
                message = "Insufficient gradient data."
            elif min_rms < 1e-5 or min_ur < 1e-4:
                # Vanishing: RMS < 1e-5 OR update ratio < 1e-4 (matches slide 5F).
                worst_layer = (
                    stats.sort("rms").row(0, named=True)["layer"]
                    if min_rms < 1e-5
                    else stats.sort("ur").row(0, named=True)["layer"]
                )
                severity = "CRITICAL"
                message = (
                    f"Vanishing gradients at '{worst_layer}' — "
                    f"min RMS = {min_rms:.2e}, min update_ratio = {min_ur:.2e}. "
                    "Fix: verify pre-norm layout (LayerNorm/RMSNorm before block), "
                    "add residual connections, switch to GELU/SiLU, or use Kaiming init."
                )
            elif max_rms > 1e-2 or max_ur > 0.1:
                # Exploding: RMS > 1e-2 OR update ratio > 0.1 (matches slide 5F).
                worst_layer = (
                    stats.sort("rms", descending=True).row(0, named=True)["layer"]
                    if max_rms > 1e-2
                    else stats.sort("ur", descending=True).row(0, named=True)["layer"]
                )
                severity = "CRITICAL"
                message = (
                    f"Exploding gradients at '{worst_layer}' — "
                    f"max RMS = {max_rms:.2e}, max update_ratio = {max_ur:.2e}. "
                    "Fix: add gradient clipping (or adaptive: ZClip/AGC), reduce LR, "
                    "verify initialization (Kaiming for ReLU, Xavier for Tanh)."
                )
            elif max_rms / max(min_rms, 1e-12) > 1e3:
                severity = "WARNING"
                message = (
                    f"Large RMS spread across layers (max/min = "
                    f"{max_rms / min_rms:.1e}). Deep layers may be learning unevenly."
                )
            else:
                severity = "HEALTHY"
                message = (
                    f"Gradient flow OK (RMS range {min_rms:.2e}–{max_rms:.2e}, "
                    f"update_ratio range {min_ur:.2e}–{max_ur:.2e})."
                )
            findings["gradient_flow"] = {"severity": severity, "message": message}
        elif gdf.height:
            # Legacy path for dataframes without the new columns.
            findings["gradient_flow"] = {
                "severity": "UNKNOWN",
                "message": (
                    "grad_rms field missing — re-run with the current library "
                    "version to get scale-invariant diagnosis."
                ),
            }
        else:
            findings["gradient_flow"] = {
                "severity": "UNKNOWN",
                "message": "No gradient tracking enabled — call track_gradients().",
            }

        # 2. Dead neurons / saturation — uses ACTIVATION-TYPE-AWARE
        # inactivity_fraction from _act_log (ReLU: |x|<eps; Tanh: |x|>0.99;
        # Sigmoid: |x|>0.99 or |x|<0.01). This catches near-dead ReLU channels
        # that the legacy exact-zero check misses post-BatchNorm.
        adf = self.activations_df()
        if adf.height and "inactivity_fraction" in adf.columns:
            # Aggregate mean inactivity per layer (averaged across batches).
            agg = (
                adf.filter(pl.col("act_kind") != "other")
                .group_by("layer")
                .agg(
                    pl.col("inactivity_fraction").mean().alias("mean_inactive"),
                    pl.col("act_kind").first().alias("kind"),
                )
                .sort("mean_inactive", descending=True)
            )
            if agg.height:
                worst = agg.row(0, named=True)
                thr = self.dead_neuron_threshold
                if worst["mean_inactive"] > thr:
                    kind = worst["kind"]
                    if kind == "relu":
                        label = "dead neurons"
                        fix = "Switch to GELU/LeakyReLU or re-initialise with Kaiming."
                    elif kind == "tanh":
                        label = "saturated (|x|>0.99)"
                        fix = "Reduce LR, add LayerNorm before, or switch to GELU."
                    elif kind == "sigmoid":
                        label = "saturated (|x|>0.99 or |x|<0.01)"
                        fix = "Reduce LR, add BN/LN, or switch to softmax+CE if classification."
                    else:
                        label = "inactive"
                        fix = "Investigate activation distribution."
                    findings["dead_neurons"] = {
                        "severity": "WARNING",
                        "message": (
                            f"'{worst['layer']}' ({kind}): "
                            f"{worst['mean_inactive']:.0%} {label}. {fix}"
                        ),
                    }
                else:
                    findings["dead_neurons"] = {
                        "severity": "HEALTHY",
                        "message": (
                            f"All {agg.height} activation layers healthy "
                            f"(worst: {worst['layer']} at "
                            f"{worst['mean_inactive']:.0%} inactive, below "
                            f"{thr:.0%} threshold)."
                        ),
                    }
            else:
                findings["dead_neurons"] = {
                    "severity": "UNKNOWN",
                    "message": "No activation layers tracked — call track_activations().",
                }
        else:
            findings["dead_neurons"] = {
                "severity": "UNKNOWN",
                "message": "No activation tracking enabled — call track_activations().",
            }

        # 3. Loss trend
        edf = self.epochs_df()
        if edf.height >= 3 and edf["val_loss"].is_not_null().any():
            overfit = self._detect_overfit_epoch()
            train_trend = self._slope(edf["train_loss"].to_list())
            if overfit is not None:
                severity = "WARNING"
                message = (
                    f"Overfitting suspected at epoch {overfit} "
                    "(val loss rising while train loss falls). "
                    "Consider dropout, weight decay, or early stopping."
                )
            elif train_trend > -1e-4 and edf.height >= 5:
                severity = "WARNING"
                message = (
                    f"Underfitting — train loss slope {train_trend:.2e}/epoch. "
                    "Consider a larger model, more epochs, or higher LR."
                )
            else:
                severity = "HEALTHY"
                message = f"Loss converging (train slope {train_trend:.2e}/epoch)."
            findings["loss_trend"] = {"severity": severity, "message": message}
        else:
            findings["loss_trend"] = {
                "severity": "UNKNOWN",
                "message": "Need at least 3 epochs with val_loss for trend analysis.",
            }

        # Human-readable summary
        print("\n" + "═" * 64)
        print("  DL Diagnostics Report — Prescription Pad")
        print("═" * 64)
        icons = {
            "HEALTHY": "✓",
            "WARNING": "!",
            "CRITICAL": "X",
            "UNKNOWN": "?",
        }
        titles = {
            "gradient_flow": "Gradient flow",
            "dead_neurons": "Dead neurons ",
            "loss_trend": "Loss trend   ",
        }
        for key, label in titles.items():
            f = findings[key]
            print(
                f"  [{icons[f['severity']]}] {label} ({f['severity']}): {f['message']}"
            )
        print("═" * 64 + "\n")
        return findings

    # ── Grad-CAM ──────────────────────────────────────────────────────────

    def grad_cam(
        self,
        input_tensor: torch.Tensor,
        target_class: int,
        layer_name: str,
    ) -> torch.Tensor:
        """Compute a Grad-CAM heatmap for ``target_class`` from ``layer_name``.

        Args:
            input_tensor: Input batch ``(N, C, H, W)`` or ``(N, C, L)``.
            target_class: Target class index to explain.
            layer_name: ``model.named_modules()`` key of the conv layer to
                attribute. Usually the last convolutional layer.

        Returns:
            Heatmap tensor with shape matching the spatial dims of the tracked
            layer's output (``(N, H', W')`` for 2D inputs).

        Raises:
            ValueError: If ``layer_name`` is not found in the model.
        """
        target_module: nn.Module | None = None
        for name, module in self.model.named_modules():
            if name == layer_name:
                target_module = module
                break
        if target_module is None:
            raise ValueError(
                f"Layer '{layer_name}' not found in model. "
                f"Available: {[n for n, _ in self.model.named_modules() if n][:10]}..."
            )

        self._gradcam_activation = None
        self._gradcam_gradient = None

        def fwd_hook(_mod: nn.Module, _inp: Any, out: torch.Tensor) -> None:
            self._gradcam_activation = out.detach()

        def bwd_hook(_mod: nn.Module, _gi: Any, go: Any) -> None:
            # go is a tuple — first element is d(output)/d(module-output)
            self._gradcam_gradient = go[0].detach()

        h1 = target_module.register_forward_hook(fwd_hook)
        h2 = target_module.register_full_backward_hook(bwd_hook)
        self._handles.grad_cam.extend([h1, h2])

        # Preserve caller's train/eval state — critical for mid-training use.
        was_training = self.model.training

        try:
            self.model.eval()
            inp = input_tensor.to(self.device)
            logits = self.model(inp)
            if logits.ndim != 2:
                raise ValueError(
                    f"grad_cam expects classification logits (N, C); got {logits.shape}"
                )
            self.model.zero_grad(set_to_none=True)
            one_hot = torch.zeros_like(logits)
            one_hot[:, target_class] = 1.0
            logits.backward(gradient=one_hot, retain_graph=False)

            if self._gradcam_activation is None or self._gradcam_gradient is None:
                raise RuntimeError(
                    "Grad-CAM hooks did not fire — layer may be unreachable "
                    "from the forward path."
                )
            activations = self._gradcam_activation  # (N, K, ...)
            gradients = self._gradcam_gradient  # (N, K, ...)
            # Global-average-pool gradients over spatial dims to get per-channel weights.
            spatial_dims = tuple(range(2, gradients.ndim))
            weights = gradients.mean(dim=spatial_dims, keepdim=True)  # (N, K, 1, ...)
            cam = (weights * activations).sum(dim=1)  # (N, ...)
            cam = torch.relu(cam)
            # Normalise per-sample to [0, 1].
            flat = cam.flatten(start_dim=1)
            mins = flat.min(dim=1, keepdim=True).values
            maxs = flat.max(dim=1, keepdim=True).values
            denom = (maxs - mins).clamp(min=1e-8)
            flat = (flat - mins) / denom
            cam = flat.view_as(cam)
            return cam.cpu()
        finally:
            # Restore caller's train/eval state BEFORE removing hooks, so
            # downstream errors in hook cleanup don't leave model in eval mode.
            if was_training:
                self.model.train()
            h1.remove()
            h2.remove()
            # Remove from bookkeeping too so detach() stays idempotent.
            self._handles.grad_cam = [
                h for h in self._handles.grad_cam if h is not h1 and h is not h2
            ]
            self._gradcam_activation = None
            self._gradcam_gradient = None

    # ── LR range test (static) ────────────────────────────────────────────

    @staticmethod
    def lr_range_test(
        model: nn.Module,
        dataloader: DataLoader,
        *,
        loss_fn: nn.Module | None = None,
        optimizer_cls: type[torch.optim.Optimizer] = torch.optim.SGD,
        lr_min: float = 1e-7,
        lr_max: float = 10.0,
        steps: int = 200,
        device: torch.device | None = None,
        batch_adapter: Callable[[Any], tuple[torch.Tensor, torch.Tensor]] | None = None,
        safety_divisor: float = 10.0,
    ) -> dict[str, Any]:
        """Leslie Smith LR range test (with fastai-style safe-LR recipe).

        Trains for ``steps`` batches while exponentially increasing LR from
        ``lr_min`` to ``lr_max``. Smooths the loss curve with EMA (beta=0.98)
        before finding the steepest-descent point — this matches fastai's
        ``lr_find()`` and avoids picking a single lucky batch in the tail.

        **Critical**: returns BOTH ``min_loss_lr`` (steepest descent) AND
        ``safe_lr = min_loss_lr / safety_divisor`` (default 10). Use ``safe_lr``
        in your optimizer — ``min_loss_lr`` is the edge of instability.

        The model's weights are saved before the test and **restored** on exit
        (via state_dict deepcopy), so calling this does not corrupt your model.

        Args:
            model: The model to probe. Weights are restored after return.
            dataloader: Any DataLoader. By default yields ``(inputs, targets)``
                tuples; pass ``batch_adapter`` for HF/SSL batch formats.
            loss_fn: Loss function (REQUIRED — no silent default).
            optimizer_cls: Optimizer class (default SGD).
            lr_min, lr_max, steps: Sweep configuration.
            device: Override compute device (default: model's current device).
            batch_adapter: Callable ``batch -> (x, y)``. Default handles
                tuple/list; pass a lambda for dict-shaped batches (e.g. HF).
            safety_divisor: Divide steepest-descent LR by this to get safe LR
                (fastai default: 10).

        Returns:
            ``{"safe_lr": float,              # use this in your optimizer
                "min_loss_lr": float,          # steepest descent (edge of instability)
                "divergence_lr": float,        # where smoothed loss > 4× min
                "lrs": [...], "losses": [...], "losses_smooth": [...],
                "figure": go.Figure}``
        """
        if steps < 2:
            raise ValueError("steps must be >= 2")
        if not 0 < lr_min < lr_max:
            raise ValueError("require 0 < lr_min < lr_max")
        if loss_fn is None:
            raise ValueError(
                "loss_fn is required — no silent default. "
                "Pass nn.CrossEntropyLoss() for classification or nn.MSELoss() for regression."
            )

        import copy as _copy

        # Device: honor model's current device unless overridden.
        dev = device or next(model.parameters()).device
        if device is not None:
            model = model.to(dev)

        # Save weights for restoration.
        saved_state = _copy.deepcopy(model.state_dict())

        # Default batch adapter handles tuple/list; raises loudly on dicts.
        def _default_adapter(batch: Any) -> tuple[torch.Tensor, torch.Tensor]:
            if isinstance(batch, (tuple, list)) and len(batch) >= 2:
                return batch[0], batch[1]
            raise ValueError(
                "lr_range_test got a non-(x,y) batch. Pass batch_adapter=... "
                "for HuggingFace-style dict batches or SSL single-tensor batches."
            )

        adapter = batch_adapter or _default_adapter

        try:
            optimizer = optimizer_cls(model.parameters(), lr=lr_min)
            gamma = (lr_max / lr_min) ** (1.0 / (steps - 1))
            lrs: list[float] = []
            losses: list[float] = []
            running_min = float("inf")  # O(1) running min, not O(n)
            model.train()
            data_iter = iter(dataloader)
            for step in range(steps):
                try:
                    batch = next(data_iter)
                except StopIteration:
                    data_iter = iter(dataloader)
                    batch = next(data_iter)
                x, y = adapter(batch)
                x, y = x.to(dev), y.to(dev)
                for pg in optimizer.param_groups:
                    pg["lr"] = lr_min * (gamma**step)
                optimizer.zero_grad(set_to_none=True)
                logits = model(x)
                loss = loss_fn(logits, y)
                loss.backward()
                optimizer.step()
                cur_lr = optimizer.param_groups[0]["lr"]
                cur_loss = float(loss.item())
                lrs.append(cur_lr)
                losses.append(cur_loss)
                if cur_loss < running_min:
                    running_min = cur_loss
                if not math.isfinite(cur_loss) or cur_loss > 10 * running_min:
                    logger.info(
                        "dldiagnostics.lr_range_test.diverged",
                        extra={"step": step, "lr": cur_lr, "loss": cur_loss},
                    )
                    break
        finally:
            # Always restore weights — no silent corruption.
            model.load_state_dict(saved_state)

        # EMA-smoothed losses (fastai convention, beta=0.98).
        beta = 0.98
        losses_smooth: list[float] = []
        ema = 0.0
        for i, ell in enumerate(losses):
            ema = beta * ema + (1 - beta) * ell
            # Bias-correct the EMA.
            losses_smooth.append(ema / (1 - beta ** (i + 1)))

        # min_loss_lr = LR at the steepest-descent point of SMOOTHED loss.
        min_loss_lr = lr_min
        divergence_lr = lr_max
        if len(losses_smooth) >= 3:
            dl = np.diff(np.array(losses_smooth))
            idx = int(np.argmin(dl))
            min_loss_lr = lrs[idx]
            # Divergence = first point where smoothed loss > 4× its minimum.
            min_smooth = min(losses_smooth)
            for i, s in enumerate(losses_smooth):
                if s > 4 * min_smooth and i > idx:
                    divergence_lr = lrs[i]
                    break

        safe_lr = min_loss_lr / safety_divisor

        fig = go.Figure()
        fig.add_trace(
            go.Scatter(
                x=lrs,
                y=losses,
                mode="lines",
                name="raw loss",
                line=dict(color="lightgray"),
            )
        )
        fig.add_trace(
            go.Scatter(
                x=lrs,
                y=losses_smooth,
                mode="lines",
                name="smoothed",
                line=dict(color="#0D9488", width=2),
            )
        )
        fig.add_vline(
            x=safe_lr,
            line=dict(color="#22C55E", dash="dash"),
            annotation_text=f"safe_lr = {safe_lr:.2e}",
        )
        fig.add_vline(
            x=min_loss_lr,
            line=dict(color="#F59E0B", dash="dot"),
            annotation_text=f"min_loss_lr = {min_loss_lr:.2e}",
        )
        fig.update_layout(
            title="LR Range Test (smoothed) — use safe_lr, not min_loss_lr",
            xaxis_title="learning rate",
            yaxis_title="loss",
            xaxis_type="log",
            template="plotly_white",
        )
        logger.info(
            "dldiagnostics.lr_range_test.ok",
            extra={
                "safe_lr": safe_lr,
                "min_loss_lr": min_loss_lr,
                "divergence_lr": divergence_lr,
                "steps_run": len(lrs),
            },
        )
        return {
            "safe_lr": safe_lr,
            "min_loss_lr": min_loss_lr,
            "divergence_lr": divergence_lr,
            "suggested_lr": safe_lr,  # backwards-compat alias
            "lrs": lrs,
            "losses": losses,
            "losses_smooth": losses_smooth,
            "figure": fig,
        }

    # ── Hook factories (internal) ─────────────────────────────────────────

    def _make_grad_hook(self, name: str):
        """Scale-invariant gradient tracking.

        Records three metrics per layer per batch:
          - grad_norm: raw L2 norm (preserved for backwards compatibility)
          - grad_rms: per-element RMS = ‖∇‖ / sqrt(numel) — scale-invariant,
            comparable across layers of any size. This is what LLM training
            dashboards (W&B, TensorBoard) actually display.
          - update_ratio: ‖∇W‖ / ‖W‖ — the "effective LR" per layer.

        Casts to fp32 before reduction so BF16/FP16 gradients don't silently
        produce inf/NaN.
        """
        # Look up the parameter tensor for update-ratio computation.
        try:
            param = dict(self.model.named_parameters())[name]
        except KeyError:
            param = None

        def _hook(grad: torch.Tensor) -> None:
            try:
                # Handle sparse gradients (e.g. nn.Embedding(sparse=True)).
                g = grad.coalesce().values() if grad.is_sparse else grad
                # Cast to fp32 to avoid BF16/FP16 inf.
                g_f32 = g.detach().float()
                l2 = float(g_f32.norm(p=2).item())
                numel = max(g_f32.numel(), 1)
                rms = l2 / (numel**0.5)
                # Update ratio — use detached param weight norm.
                if param is not None:
                    p_norm = float(param.detach().float().norm(p=2).item())
                    upd_ratio = l2 / max(p_norm, 1e-12)
                else:
                    upd_ratio = 0.0
            except Exception as exc:  # pragma: no cover - defensive
                logger.warning(
                    "dldiagnostics.grad_hook.error",
                    extra={"layer": name, "error": str(exc)},
                )
                return
            self._grad_log.append(
                {
                    "batch": self._batch_idx,
                    "layer": name,
                    "grad_norm": l2,  # preserved for compatibility
                    "grad_rms": rms,  # scale-invariant
                    "update_ratio": upd_ratio,  # ‖∇W‖ / ‖W‖
                }
            )

        return _hook

    def _make_act_hook(self, name: str):
        """Activation statistics. Casts to fp32 to avoid BF16/FP16 overflow.

        The `inactivity_fraction` field measures activation-type-appropriate
        pathology:
          - ReLU / GELU / SiLU: fraction with |x| < 1e-6 (dead neurons)
          - Tanh: fraction with |x| > 0.99 (saturated)
          - Sigmoid: fraction with |x| > 0.99 OR |x| < 0.01 (saturated)
          - Others: 0 (no pathology definition)

        The legacy `dead_fraction` field (exact-zero count) is preserved for
        backwards compatibility but is only meaningful for ReLU.
        """
        # Determine activation type from module class name for dispatching.
        act_kind = self._classify_activation_module(name)

        def _hook(_module: nn.Module, _inp: Any, output: torch.Tensor) -> None:
            if not isinstance(output, torch.Tensor) or output.numel() == 0:
                return
            # Cast to fp32 for numerical safety (BF16/FP16 overflow on .std()).
            out = output.detach().float()
            try:
                mean = float(out.mean().item())
                std = float(out.std().item()) if out.numel() > 1 else 0.0
                mn = float(out.min().item())
                mx = float(out.max().item())
                legacy_dead = float((out == 0).float().mean().item())
                # Activation-aware inactivity/saturation metric.
                if act_kind == "relu":
                    inactivity = float((out.abs() < 1e-6).float().mean().item())
                elif act_kind == "tanh":
                    inactivity = float((out.abs() > 0.99).float().mean().item())
                elif act_kind == "sigmoid":
                    inactivity = float(
                        ((out > 0.99) | (out < 0.01)).float().mean().item()
                    )
                else:
                    inactivity = 0.0
            except RuntimeError:
                # Non-numeric tensor (e.g. mixed dtype). Skip silently.
                return
            # Guard against NaN/inf leaking into logs.
            for val_name, val in (("mean", mean), ("std", std)):
                if val != val or val in (float("inf"), float("-inf")):
                    logger.warning(
                        "dldiagnostics.act_hook.nonfinite",
                        extra={"layer": name, "field": val_name},
                    )
                    return
            self._act_log.append(
                {
                    "batch": self._batch_idx,
                    "layer": name,
                    "act_kind": act_kind,
                    "mean": mean,
                    "std": std,
                    "min": mn,
                    "max": mx,
                    "dead_fraction": legacy_dead,
                    "inactivity_fraction": inactivity,
                }
            )

        return _hook

    def _classify_activation_module(self, name: str) -> str:
        """Return one of 'relu', 'tanh', 'sigmoid', 'other' for a module name."""
        try:
            mod = dict(self.model.named_modules())[name]
        except KeyError:
            return "other"
        cls = type(mod).__name__.lower()
        if any(k in cls for k in ("relu", "gelu", "silu", "swish", "elu")):
            return "relu"
        if "tanh" in cls:
            return "tanh"
        if "sigmoid" in cls:
            return "sigmoid"
        return "other"

    def _make_dead_hook(self, name: str):
        def _hook(_module: nn.Module, _inp: Any, output: torch.Tensor) -> None:
            if not isinstance(output, torch.Tensor) or output.numel() == 0:
                return
            out = output.detach()
            # Collapse all non-channel dims. Convention: channel dim = 1 for
            # Conv outputs (N, C, ...); for Linear, output is (N, F) — here
            # we treat dim 1 as "neurons" which matches both shapes.
            if out.ndim < 2:
                return
            reduce_dims = tuple(d for d in range(out.ndim) if d != 1)
            fired = (out != 0).any(dim=reduce_dims).float().cpu()
            if name not in self._firing_counts:
                self._firing_counts[name] = torch.zeros_like(fired)
                self._firing_samples[name] = 0
            self._firing_counts[name] += fired
            self._firing_samples[name] += 1
            # Keep memory bounded by decaying old counts when window exceeded.
            if self._firing_samples[name] > self.window:
                scale = self.window / self._firing_samples[name]
                self._firing_counts[name] = self._firing_counts[name] * scale
                self._firing_samples[name] = self.window

        return _hook

    # ── Internal analysis helpers ─────────────────────────────────────────

    @staticmethod
    def _slope(series: list[float]) -> float:
        """Least-squares slope of a 1D series vs its index (ignores NaN)."""
        xs = np.arange(len(series), dtype=float)
        ys = np.asarray(series, dtype=float)
        mask = np.isfinite(ys)
        if mask.sum() < 2:
            return 0.0
        xs, ys = xs[mask], ys[mask]
        return float(np.polyfit(xs, ys, 1)[0])

    def _detect_overfit_epoch(self) -> int | None:
        """Return epoch index where val loss starts rising while train falls."""
        edf = self.epochs_df()
        if edf.height < 3:
            return None
        train = edf["train_loss"].to_list()
        val = edf["val_loss"].to_list()
        for i in range(2, len(val)):
            v_now, v_prev = val[i], val[i - 1]
            t_now, t_prev = train[i], train[i - 1]
            if any(
                x is None or not math.isfinite(x)
                for x in (v_now, v_prev, t_now, t_prev)
            ):
                continue
            if v_now > v_prev and t_now < t_prev:
                # Confirm trend persists one step back if available.
                return i
        return None


# ════════════════════════════════════════════════════════════════════════
# Standalone diagnostic checkpoint — for use AFTER a training loop
# ════════════════════════════════════════════════════════════════════════


def run_diagnostic_checkpoint(
    model: nn.Module,
    dataloader: DataLoader,
    loss_fn: Callable[..., Any],
    *,
    title: str = "Model",
    n_batches: int = 8,
    train_losses: list[float] | None = None,
    val_losses: list[float] | None = None,
    show: bool = True,
    batch_adapter: Callable[[Any], tuple[torch.Tensor, ...]] | None = None,
) -> tuple[DLDiagnostics, dict[str, Any]]:
    """Run a short instrumented diagnostic pass on a TRAINED model.

    Use this between the Train and Visualise phases of an exercise. It
    attaches the four diagnostic instruments (gradients, activations,
    dead-neurons, scalar history) to the model, runs ``n_batches`` of
    forward-backward passes to populate the history, replays any
    epoch-level losses you collected during real training, and prints the
    Prescription Pad with auto-diagnosis.

    The model's weights are NOT updated — gradients are computed but no
    optimizer step is taken. The model's train/eval state is preserved.

    Args:
        model: A trained ``nn.Module``.
        dataloader: Any DataLoader whose batches the loss function accepts.
        loss_fn: Callable. The default contract is
            ``loss_fn(model, batch) -> (scalar_loss, extras_dict)`` matching
            the M5 exercise convention. Pass ``batch_adapter`` to wrap a
            different signature.
        title: Human-readable name printed in the dashboard title.
        n_batches: How many batches to instrument (default 8 — enough for
            a stable mean per layer without slowing the exercise down).
        train_losses: Optional list of per-epoch train losses captured
            during the actual training run. Replayed into the dashboard's
            stethoscope view so students see the real loss trajectory, not
            just the diagnostic-pass losses.
        val_losses: Optional list of per-epoch validation losses, same
            length as ``train_losses``.
        show: If ``True``, calls ``.show()`` on the dashboard figure.
        batch_adapter: Optional ``batch -> (loss_fn_args...)`` translator
            for non-standard batch shapes.

    Returns:
        ``(diag, findings)`` so the caller can inspect the DataFrames and
        the Prescription Pad findings dict.
    """
    if not isinstance(model, nn.Module):
        raise TypeError("run_diagnostic_checkpoint requires an nn.Module")
    if n_batches < 1:
        raise ValueError("n_batches must be >= 1")

    diag = DLDiagnostics(model)
    diag.track_gradients().track_activations().track_dead_neurons()

    # Replay real training history into the dashboard if provided.
    if train_losses or val_losses:
        n_epochs = max(len(train_losses or []), len(val_losses or []))
        for i in range(n_epochs):
            tl = (
                float(train_losses[i])
                if train_losses and i < len(train_losses)
                else None
            )
            vl = float(val_losses[i]) if val_losses and i < len(val_losses) else None
            # Synthesise one batch entry per epoch so the per-batch stethoscope
            # has data to plot — students still see the real epoch losses.
            if tl is not None:
                diag.record_batch(loss=tl)
            diag.record_epoch(train_loss=tl, val_loss=vl)

    was_training = model.training
    try:
        model.train()  # Enable gradients + activation hooks.
        seen = 0
        for batch in dataloader:
            if seen >= n_batches:
                break
            try:
                if batch_adapter is not None:
                    args = batch_adapter(batch)
                    loss_out = loss_fn(model, *args)
                else:
                    loss_out = loss_fn(model, batch)
                # Convention: M5 loss_fn returns (loss, extras_dict). Allow
                # either a bare tensor or a tuple.
                if isinstance(loss_out, tuple):
                    loss = loss_out[0]
                else:
                    loss = loss_out
                if not isinstance(loss, torch.Tensor):
                    raise TypeError(
                        f"loss_fn returned {type(loss).__name__}; expected Tensor"
                    )
                model.zero_grad(set_to_none=True)
                loss.backward()
                # NOTE: no optimizer.step() — diagnostic pass is read-only.
                diag.record_batch(loss=float(loss.item()))
            except Exception as exc:  # pragma: no cover - student loop variability
                logger.warning(
                    "dldiagnostics.checkpoint.batch_skipped",
                    extra={"batch_idx": seen, "error": str(exc)},
                )
            seen += 1
    finally:
        if not was_training:
            model.eval()

    # Render the dashboard and the Prescription Pad.
    fig = diag.plot_training_dashboard()
    fig.update_layout(title=f"{title} — Diagnostic Dashboard (Vital Signs)")
    if show:
        try:
            fig.show()
        except Exception as exc:  # pragma: no cover - no display in CI
            logger.info(
                "dldiagnostics.checkpoint.show_skipped",
                extra={"error": str(exc)},
            )

    findings = diag.report()
    return diag, findings


def diagnose_classifier(
    model: nn.Module,
    dataloader: DataLoader,
    *,
    title: str = "Classifier",
    n_batches: int = 8,
    train_losses: list[float] | None = None,
    val_losses: list[float] | None = None,
    show: bool = True,
    forward_returns_tuple: bool = False,
) -> tuple[DLDiagnostics, dict[str, Any]]:
    """Convenience wrapper for ``(x, y)`` cross-entropy classifiers.

    Equivalent to :func:`run_diagnostic_checkpoint` with a built-in
    ``loss_fn`` that calls ``F.cross_entropy(model(x), y)``. Use this for
    CNN, transformer, and transfer-learning exercises.

    Args:
        model: Trained classifier ``nn.Module``.
        dataloader: Yields ``(x, y)`` tuples; ``y`` is class indices.
        title: Title for the dashboard.
        n_batches: Batches to instrument.
        train_losses, val_losses: Optional epoch histories to replay.
        show: Show the figure inline.
        forward_returns_tuple: If ``True``, ``model(x)`` returns
            ``(logits, ...)`` (e.g. attention models) — only the first
            element is used as logits.

    Returns:
        ``(diag, findings)``
    """
    import torch.nn.functional as F  # local — torch already imported

    def _cls_loss(m: nn.Module, batch: Any) -> torch.Tensor:
        x, y = batch[0], batch[1]
        out = m(x)
        logits = out[0] if forward_returns_tuple else out
        return F.cross_entropy(logits, y)

    return run_diagnostic_checkpoint(
        model,
        dataloader,
        _cls_loss,
        title=title,
        n_batches=n_batches,
        train_losses=train_losses,
        val_losses=val_losses,
        show=show,
    )


def diagnose_regressor(
    model: nn.Module,
    dataloader: DataLoader,
    *,
    title: str = "Regressor",
    n_batches: int = 8,
    train_losses: list[float] | None = None,
    val_losses: list[float] | None = None,
    show: bool = True,
    forward_returns_tuple: bool = False,
) -> tuple[DLDiagnostics, dict[str, Any]]:
    """Convenience wrapper for ``(x, y)`` MSE regressors (RNN exercises).

    Equivalent to :func:`run_diagnostic_checkpoint` with a built-in
    ``loss_fn`` that calls ``F.mse_loss(model(x), y)``.

    Args:
        forward_returns_tuple: Set ``True`` for attention models that
            return ``(predictions, attn_weights)``.
    """
    import torch.nn.functional as F

    def _reg_loss(m: nn.Module, batch: Any) -> torch.Tensor:
        x, y = batch[0], batch[1]
        out = m(x)
        pred = out[0] if forward_returns_tuple else out
        return F.mse_loss(pred, y)

    return run_diagnostic_checkpoint(
        model,
        dataloader,
        _reg_loss,
        title=title,
        n_batches=n_batches,
        train_losses=train_losses,
        val_losses=val_losses,
        show=show,
    )


__all__ = [
    "DLDiagnostics",
    "run_diagnostic_checkpoint",
    "diagnose_classifier",
    "diagnose_regressor",
]

# ── quiz_harness.py ──
"""Quiz harness for MLFP05 — shared checkers and training helpers.

All functions in this module are used by both the student notebook and the
instructor answer key so the grading bar is identical in both.

Design principles:

* Deterministic seeds so student & instructor see the same accuracy bands.
* Works on CPU, MPS (Apple Silicon), and CUDA (Colab T4).
* Download-once caching: MNIST / Fashion-MNIST live under ``/content/data/``
  on Colab and ``data/mlfp05/`` locally so re-runs are fast.
* Diagnostic thresholds for Q5 were calibrated by running the TARGET
  architecture and capturing its actual ``report()`` verdicts (see the
  module docstring in ``mlfp05_quiz_solutions.ipynb``).
"""

import math
import time
from pathlib import Path
from typing import Any

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision import datasets, transforms


# ── Device detection ──────────────────────────────────────────────────────
def pick_device() -> torch.device:
    """Return CUDA if available, else MPS (Apple), else CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


# ── Data root resolution ──────────────────────────────────────────────────
def _resolve_data_root(kind: str) -> Path:
    """Resolve the cache directory for a torchvision dataset.

    Priority:
      1. ``/content/data/{kind}`` (Colab convention)
      2. ``<repo_root>/data/mlfp05/{kind}`` (local convention)
      3. fallback: ``./data/{kind}``

    ``kind`` is one of ``"mnist"`` / ``"fashion_mnist"``.
    """
    colab_root = Path("/content/data") / kind
    if Path("/content").exists():
        colab_root.mkdir(parents=True, exist_ok=True)
        return colab_root

    # Walk up to find a repo containing modules/mlfp05/
    here = Path(__file__).resolve()
    for ancestor in here.parents:
        candidate = ancestor / "data" / "mlfp05" / kind
        if (ancestor / "modules" / "mlfp05").exists():
            candidate.mkdir(parents=True, exist_ok=True)
            return candidate

    fallback = Path.cwd() / "data" / kind
    fallback.mkdir(parents=True, exist_ok=True)
    return fallback


# ── Dataset loaders (cached) ──────────────────────────────────────────────
_MNIST_STATS = ((0.1307,), (0.3081,))
_FASHION_STATS = ((0.2860,), (0.3530,))


def load_mnist(batch_size: int = 128) -> tuple[DataLoader, DataLoader]:
    """Return (train_loader, test_loader) for MNIST. Downloads once, caches."""
    transform = transforms.Compose(
        [transforms.ToTensor(), transforms.Normalize(*_MNIST_STATS)]
    )
    root = _resolve_data_root("mnist")
    train = datasets.MNIST(
        root=str(root), train=True, download=True, transform=transform
    )
    test = datasets.MNIST(
        root=str(root), train=False, download=True, transform=transform
    )
    return (
        DataLoader(train, batch_size=batch_size, shuffle=True, num_workers=0),
        DataLoader(test, batch_size=512, shuffle=False, num_workers=0),
    )


def load_fashion_mnist(batch_size: int = 128) -> tuple[DataLoader, DataLoader]:
    """Return (train_loader, val_loader) for Fashion-MNIST. Downloads once."""
    transform = transforms.Compose(
        [transforms.ToTensor(), transforms.Normalize(*_FASHION_STATS)]
    )
    root = _resolve_data_root("fashion_mnist")
    train = datasets.FashionMNIST(
        root=str(root), train=True, download=True, transform=transform
    )
    test = datasets.FashionMNIST(
        root=str(root), train=False, download=True, transform=transform
    )
    return (
        DataLoader(train, batch_size=batch_size, shuffle=True, num_workers=0),
        DataLoader(test, batch_size=512, shuffle=False, num_workers=0),
    )


# ── Q3 synthetic sequence dataset ─────────────────────────────────────────
def make_q3_dataset(
    n_train: int = 4096,
    n_test: int = 1024,
    seq_len: int = 20,
    seed: int = 42,
) -> tuple[DataLoader, DataLoader]:
    """Binary-sequence task: label = 1 iff more 1s are in the FIRST half.

    Each sequence is ``seq_len`` Bernoulli(0.5) draws. Tie ⇒ label 0. The
    task is *trivially* linearly separable in sufficient statistics but
    requires a model that can distinguish positional information — a
    bag-of-words MLP will score ≈ 50%, an LSTM should score >>85%.
    """
    g = torch.Generator().manual_seed(seed)
    train_x = torch.randint(0, 2, (n_train, seq_len), generator=g).float()
    test_x = torch.randint(0, 2, (n_test, seq_len), generator=g).float()

    def _label(x: torch.Tensor) -> torch.Tensor:
        half = x.size(1) // 2
        first = x[:, :half].sum(dim=1)
        second = x[:, half:].sum(dim=1)
        return (first > second).long()

    train_y = _label(train_x)
    test_y = _label(test_x)

    # Shape as (B, T, 1) so nn.LSTM(input_size=1) consumes directly.
    train_x = train_x.unsqueeze(-1)
    test_x = test_x.unsqueeze(-1)

    train = TensorDataset(train_x, train_y)
    test = TensorDataset(test_x, test_y)
    return (
        DataLoader(train, batch_size=128, shuffle=True, num_workers=0),
        DataLoader(test, batch_size=512, shuffle=False, num_workers=0),
    )


# ── Generic eval helpers ──────────────────────────────────────────────────
@torch.no_grad()
def accuracy(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    """Return classification accuracy ∈ [0,1] on the given loader."""
    model.eval()
    model.to(device)
    correct = 0
    total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.numel()
    return correct / max(total, 1)


def train_classifier(
    model: nn.Module,
    train_loader: DataLoader,
    test_loader: DataLoader,
    device: torch.device,
    *,
    epochs: int = 5,
    lr: float = 1e-3,
    label: str = "model",
) -> float:
    """Standard Adam + CE training loop. Returns final test accuracy.

    Used by Q1/Q2/Q3. Prints one line per epoch so students see progress.
    """
    model.to(device)
    optim = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    t0 = time.monotonic()
    for epoch in range(1, epochs + 1):
        model.train()
        running = 0.0
        n_batches = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optim.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            optim.step()
            running += loss.item()
            n_batches += 1
        test_acc = accuracy(model, test_loader, device)
        print(
            f"  [{label}] epoch {epoch}/{epochs}  "
            f"train_loss={running/max(n_batches,1):.4f}  "
            f"test_acc={test_acc:.4f}  "
            f"elapsed={time.monotonic()-t0:.1f}s"
        )
    return test_acc


# ── Q1–Q3 checkers ────────────────────────────────────────────────────────
def check_q1(test_acc: float) -> tuple[bool, float, str]:
    """Q1 passes at >95% MNIST test accuracy after 5 epochs."""
    threshold = 0.95
    passed = test_acc >= threshold
    msg = (
        f"Q1 MLP — test acc = {test_acc:.4f}  "
        f"(threshold ≥ {threshold}) — {'PASS' if passed else 'FAIL'}"
    )
    return passed, test_acc, msg


def check_q2(test_acc: float) -> tuple[bool, float, str]:
    """Q2 passes at >97% MNIST test accuracy (CNN is stronger)."""
    threshold = 0.97
    passed = test_acc >= threshold
    msg = (
        f"Q2 CNN — test acc = {test_acc:.4f}  "
        f"(threshold ≥ {threshold}) — {'PASS' if passed else 'FAIL'}"
    )
    return passed, test_acc, msg


def check_q3(test_acc: float) -> tuple[bool, float, str]:
    """Q3 passes at >85% accuracy on the synthetic halves task."""
    threshold = 0.85
    passed = test_acc >= threshold
    msg = (
        f"Q3 LSTM — test acc = {test_acc:.4f}  "
        f"(threshold ≥ {threshold}) — {'PASS' if passed else 'FAIL'}"
    )
    return passed, test_acc, msg


# ── Q4 fuzzy-keyword checker ──────────────────────────────────────────────
# The Q4 setup is a CNN with stride>1 and NO padding that collapses spatial
# dims so badly the final conv feature map can become 0×0. The primary fix
# is adding padding (or reducing stride). We accept multiple phrasings.
_Q4_ISSUE_KEYWORDS: tuple[tuple[str, ...], ...] = (
    ("stride",),
    ("padding",),
    ("dimension", "dim", "spatial", "collapse", "shrink"),
)
_Q4_FIX_KEYWORDS: tuple[tuple[str, ...], ...] = (
    ("padding",),
    ("stride",),
)


def _fuzzy_match(answer: str, keyword_groups: tuple[tuple[str, ...], ...]) -> int:
    """Count how many keyword groups have at least one keyword in answer."""
    a = answer.lower()
    hits = 0
    for group in keyword_groups:
        if any(k in a for k in group):
            hits += 1
    return hits


def check_q4(answer: str) -> tuple[bool, str]:
    """Pass iff the answer mentions the issue (stride/padding) AND a fix."""
    if not isinstance(answer, str) or len(answer.strip()) < 10:
        return False, (
            "Q4 — answer too short. Explain both the diagnosed issue and "
            "the concrete fix in a sentence or two."
        )
    issue_hits = _fuzzy_match(answer, _Q4_ISSUE_KEYWORDS)
    fix_hits = _fuzzy_match(answer, _Q4_FIX_KEYWORDS)
    passed = issue_hits >= 2 and fix_hits >= 1
    msg = (
        f"Q4 — issue keywords matched: {issue_hits}/3, "
        f"fix keywords matched: {fix_hits}/2 — "
        f"{'PASS' if passed else 'FAIL'}"
    )
    if not passed:
        msg += (
            "\n      Hint: the bug involves **stride without padding** causing "
            "spatial dimensions to collapse; fix by adding padding or reducing stride."
        )
    return passed, msg


# ── Q5 training + diagnostic helper ───────────────────────────────────────
def _build_optimizer(
    model: nn.Module, lr: float, weight_decay: float
) -> torch.optim.Optimizer:
    return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)


def _warmup_factor(step: int, warmup_steps: int) -> float:
    if warmup_steps <= 0:
        return 1.0
    if step >= warmup_steps:
        return 1.0
    return (step + 1) / warmup_steps


def train_and_diagnose(
    model: nn.Module,
    *,
    train_loader: DataLoader,
    test_loader: DataLoader,
    device: torch.device,
    lr: float = 1e-3,
    weight_decay: float = 0.0,
    warmup_steps: int = 0,
    epochs: int = 3,
    verbose: bool = True,
) -> tuple[dict[str, Any], float]:
    """Train ``model`` for ``epochs`` epochs, recording DL diagnostics.

    Returns ``(findings_dict, test_accuracy)``. ``findings_dict`` is the
    dict returned by ``DLDiagnostics.report()``; each value is a dict with
    keys ``severity`` and ``message``.

    This is the Q5 primary instrument. Deliberately wraps the full TRAIN
    → DIAGNOSE → EVAL cycle so the student never touches the diagnostic
    scaffolding — only the model and hyperparameters.
    """
    # Import lazily so module import never triggers a diagnostics init on
    # import error in downstream repos without the shared package.

    model.to(device)
    optimizer = _build_optimizer(model, lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()
    base_lr = lr
    step = 0

    with DLDiagnostics(model) as diag:
        diag.track_gradients()
        diag.track_activations()
        diag.track_dead_neurons()

        for epoch in range(1, epochs + 1):
            model.train()
            running = 0.0
            n_batches = 0
            for x, y in train_loader:
                x, y = x.to(device), y.to(device)
                # LR warmup
                factor = _warmup_factor(step, warmup_steps)
                for pg in optimizer.param_groups:
                    pg["lr"] = base_lr * factor
                optimizer.zero_grad()
                logits = model(x)
                loss = loss_fn(logits, y)
                loss.backward()
                optimizer.step()
                diag.record_batch(loss=loss.item(), lr=optimizer.param_groups[0]["lr"])
                running += loss.item()
                n_batches += 1
                step += 1

            # End-of-epoch validation loss (for loss_trend diagnosis)
            model.eval()
            val_running = 0.0
            val_n = 0
            with torch.no_grad():
                for x, y in test_loader:
                    x, y = x.to(device), y.to(device)
                    val_running += loss_fn(model(x), y).item() * y.size(0)
                    val_n += y.size(0)
            val_loss = val_running / max(val_n, 1)
            diag.record_epoch(train_loss=running / max(n_batches, 1), val_loss=val_loss)
            if verbose:
                print(
                    f"  epoch {epoch}/{epochs}  "
                    f"train_loss={running/max(n_batches,1):.4f}  "
                    f"val_loss={val_loss:.4f}"
                )

        findings = diag.report()

    test_acc = accuracy(model, test_loader, device)
    if verbose:
        print(f"  final test_acc = {test_acc:.4f}")
    return findings, test_acc


# ── Q5 pass/fail decision ─────────────────────────────────────────────────
Q5_TEST_ACC_THRESHOLD = 0.82

# Verdicts we strictly require to be HEALTHY.
#
# ``gradient_flow`` is deliberately NOT in this list even though we print its
# verdict for student feedback. Rationale: the ``DLDiagnostics`` gradient_flow
# check trips CRITICAL whenever the OUTPUT layer's gradient RMS exceeds 1e-2,
# which is a fundamental property of cross-entropy on a 10-way classifier —
# ``dL/dlogits ≈ softmax − one_hot`` has RMS near 0.06, and the output linear
# layer's per-element grad RMS lands around 1.5e-2 on Fashion-MNIST at TARGET
# hyperparameters. Requiring HEALTHY here would force a fix that is not in
# fact a bug. Instead we require the two verdicts that DO differentiate the
# TARGET from the BROKEN STARTER at a 50× gap: dead_neurons and loss_trend,
# plus the downstream test accuracy.
Q5_REQUIRED_VERDICTS = ("dead_neurons", "loss_trend")

# Verdicts whose severity is surfaced in the Prescription Pad output but does
# NOT block the pass. Students see the verdict; instructors see the verdict.
Q5_ADVISORY_VERDICTS = ("gradient_flow",)


def check_q5_pass(findings: dict[str, Any], test_acc: float) -> tuple[bool, str]:
    """Pass iff the required verdicts are HEALTHY and test_acc ≥ threshold.

    Required (blocking):
        * ``dead_neurons`` == HEALTHY  (the ReLU→GELU fix)
        * ``loss_trend`` == HEALTHY    (the warmup/LR fix)
        * ``test_acc`` >= ``Q5_TEST_ACC_THRESHOLD``

    Advisory (printed but not blocking):
        * ``gradient_flow`` — the output-layer grad RMS naturally exceeds the
          library's 1e-2 threshold on a 10-way softmax head, so we surface the
          verdict for educational purposes but do not require HEALTHY.

    Threshold is 0.82 (not 0.85) because Fashion-MNIST with a plain 6-layer
    MLP caps out around 0.87 even on TARGET hyperparameters — we leave some
    headroom for run-to-run variance. Calibrated by running ``Q5TargetMLP``
    on Fashion-MNIST for 3 epochs and capturing its ``report()`` verdicts
    (see ``mlfp05_quiz_solutions.ipynb`` module docstring).
    """
    reasons: list[str] = []
    advisory_notes: list[str] = []

    # Required verdicts — blocking
    for key in Q5_REQUIRED_VERDICTS:
        sev = findings.get(key, {}).get("severity", "UNKNOWN")
        if sev != "HEALTHY":
            msg = findings.get(key, {}).get("message", "<no message>")
            reasons.append(f"  * {key}: {sev} — {msg}")

    # Advisory verdicts — surface severity without blocking
    for key in Q5_ADVISORY_VERDICTS:
        sev = findings.get(key, {}).get("severity", "UNKNOWN")
        if sev != "HEALTHY":
            msg = findings.get(key, {}).get("message", "<no message>")
            advisory_notes.append(f"  * advisory: {key}: {sev} — {msg}")

    # Test accuracy — blocking
    if test_acc < Q5_TEST_ACC_THRESHOLD:
        reasons.append(
            f"  * test_accuracy = {test_acc:.4f} is below "
            f"the required {Q5_TEST_ACC_THRESHOLD:.2f}."
        )

    if not reasons:
        msg = (
            f"Q5 PASS — dead_neurons + loss_trend HEALTHY and "
            f"test_acc = {test_acc:.4f} ≥ {Q5_TEST_ACC_THRESHOLD:.2f}.\n"
            "      Well done. Reflect: which change had the biggest impact?"
        )
        if advisory_notes:
            msg += "\n  Advisory (non-blocking):\n" + "\n".join(advisory_notes)
            msg += (
                "\n      Note: the gradient_flow CRITICAL verdict is expected "
                "at this threshold — the output layer's gradient RMS on a "
                "10-way softmax head naturally exceeds the 1e-2 library "
                "cut-off. See slide 5F for a discussion of this bias."
            )
        return True, msg

    fail_msg = "Q5 FAIL — keep iterating on Cell 1 and re-run Cell 2:\n" + "\n".join(
        reasons
    )
    if advisory_notes:
        fail_msg += "\n  Advisory (non-blocking):\n" + "\n".join(advisory_notes)
    fail_msg += (
        "\n  Hints:\n"
        "    * Exploding gradients ⇒ lower LR, add weight decay, or add warmup.\n"
        "    * Dead neurons (ReLU) ⇒ switch to GELU or apply Kaiming init.\n"
        "    * Loss oscillation ⇒ add LayerNorm, add warmup, or lower LR."
    )
    return False, fail_msg


# ── Q4 static model (used by both student & instructor notebooks) ────────
class Q4BrokenCNN(nn.Module):
    """A CNN with stride=2 and NO padding. After two such layers starting from
    28×28, spatial dims become (28-3)/2 + 1 = 13, then (13-3)/2 + 1 = 6; the
    final conv feature map (6×6) is small enough that the flatten+linear head
    still runs, but the diagnostics toolkit surfaces the unhealthy gradient
    flow + saturated activations caused by the aggressive stride.

    Students are asked to identify the issue from the diagnostics report;
    we grade on two keyword groups (stride/padding + fix).
    """

    def __init__(self) -> None:
        super().__init__()
        # Kernel 3, stride 2, NO padding — collapses spatial dimensions fast.
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=0)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=0)
        self.fc = nn.Linear(32 * 6 * 6, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        return self.fc(x.flatten(1))


# ── Q5 target (instructor answer) & student starter ──────────────────────
class Q5TargetMLP(nn.Module):
    """The TARGET architecture for Q5 (instructor answer key).

    6 linear layers with GELU activations, post-LayerNorm layout (no
    LayerNorm on the raw pixel input), dropout 0.1, Kaiming init for all
    hidden Linear layers, Xavier init for the output logit layer, and a
    SMALL non-zero bias on every parameter (including LayerNorm biases)
    so the diagnostics' ``update_ratio = ‖∇W‖/‖W‖`` stays finite at step 0.

    Trained with Adam, lr=5e-4, weight decay 1e-4, warmup 300 steps, and
    gradient clipping at 1.0 norm — the full prescription pad from deck
    slides 5C–5J. On Fashion-MNIST, 3 epochs, MPS/T4:

        * dead_neurons  = HEALTHY (~0–5% inactive on worst layer)
        * loss_trend    = HEALTHY (monotonic decrease)
        * gradient_flow = CRITICAL (advisory only — see ``check_q5_pass``
          docstring; the output layer's grad RMS exceeds the library's
          1e-2 cut-off, which is a diagnostic quirk on 10-way softmax,
          not a training pathology)
        * test_acc      ≈ 0.87 (well above the 0.82 threshold)
    """

    def __init__(self, dropout: float = 0.1) -> None:
        super().__init__()
        dims = [784, 512, 256, 128, 64, 32, 10]
        # Post-LN: first Linear takes raw input, no LayerNorm in front.
        layers: list[nn.Module] = [
            nn.Linear(dims[0], dims[1]),
            nn.GELU(),
            nn.Dropout(dropout),
        ]
        # Subsequent blocks: LayerNorm → Linear → (GELU + Dropout if not last).
        for i, (in_dim, out_dim) in enumerate(zip(dims[1:-1], dims[2:])):
            layers.append(nn.LayerNorm(in_dim))
            layers.append(nn.Linear(in_dim, out_dim))
            if i < len(dims) - 3:  # no activation on the final logit layer
                layers.append(nn.GELU())
                layers.append(nn.Dropout(dropout))
        self.net = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self) -> None:
        linears = [m for m in self.modules() if isinstance(m, nn.Linear)]
        # Hidden Linear: Kaiming (correct variance for GELU ≈ ReLU).
        for m in linears[:-1]:
            nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            nn.init.constant_(m.bias, 0.01)
        # Output Linear: Xavier, smaller variance so output-layer gradient
        # RMS stays near the 1e-2 diagnostic threshold rather than blowing past.
        nn.init.xavier_normal_(linears[-1].weight)
        nn.init.constant_(linears[-1].bias, 0.01)
        # LayerNorm: weight=1 is the default, but the default bias=0 produces
        # ``‖W‖ = 0`` and hence an infinite update_ratio on the first batch.
        # Seed the bias with a small positive value.
        for m in self.modules():
            if isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.constant_(m.bias, 0.01)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x.view(x.size(0), -1))


class Q5BrokenStarter(nn.Module):
    """The starter architecture students begin with for Q5.

    6 linear layers with ReLU, no LayerNorm, no dropout, default init.
    Combined with lr=0.1 and no warmup this produces:
        * exploding gradients (gradient_flow = CRITICAL)
        * many dead ReLU neurons (dead_neurons = WARNING)
        * oscillating loss (loss_trend may be WARNING or UNKNOWN)
        * ~0.3–0.4 Fashion-MNIST test accuracy

    Students must edit the architecture AND hyperparameters on Cell 1
    until ``check_q5_pass`` returns True on Cell 2's output.
    """

    def __init__(self) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 10),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x.view(x.size(0), -1))


__all__ = [
    "pick_device",
    "load_mnist",
    "load_fashion_mnist",
    "make_q3_dataset",
    "accuracy",
    "train_classifier",
    "train_and_diagnose",
    "check_q1",
    "check_q2",
    "check_q3",
    "check_q4",
    "check_q5_pass",
    "Q4BrokenCNN",
    "Q5TargetMLP",
    "Q5BrokenStarter",
    "Q5_TEST_ACC_THRESHOLD",
]


In [ ]:
# Shared harness imports — used by every question.
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = pick_device()
print(f'Using device: {DEVICE}')


---

## Q1 — Build a Multi-Layer Perceptron (10 min, ≥ 0.95 MNIST test accuracy)

Write an MLP with two hidden layers and train it for 5 epochs on MNIST.

**Spec**

* Input: flatten a 1×28×28 image to a 784-vector.
* Hidden: two `nn.Linear` layers with ReLU activations (sizes 256 and 128 work).
* Output: a `nn.Linear` head with 10 logits.
* Train: 5 epochs, Adam, lr=1e-3.

Grading cell calls `check_q1(final_test_accuracy)`. Threshold: **≥ 0.95**.


In [ ]:
# Q1 — write an MLP here
#
# Spec:
#   - Input: flatten a 1×28×28 image to a 784-vector.
#   - Hidden: two Linear layers with ReLU activations (256 → 128 works).
#   - Output: Linear head with 10 logits.
#
# TODO: complete Q1MLP below.

class Q1MLP(nn.Module):
    def __init__(self):
        super().__init__()
        # Hint: use nn.Sequential for a clean 2-hidden-layer MLP.
        #       Start with nn.Flatten(), then alternate Linear/ReLU.
        self.net = nn.Sequential(
            ____,                           # TODO: flatten 1×28×28 → 784
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(____, ____), nn.ReLU(),  # TODO: hidden → hidden
            nn.Linear(____, 10),            # TODO: hidden → 10 logits
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
# Q1 — train + grade (DO NOT EDIT)
torch.manual_seed(0)
q1_train, q1_test = load_mnist()
q1_acc = train_classifier(Q1MLP(), q1_train, q1_test, DEVICE, epochs=5, lr=1e-3, label='q1')
q1_pass, q1_score, q1_msg = check_q1(q1_acc)
print('\n' + q1_msg)


---

## Q2 — Build a Convolutional Classifier (15 min, ≥ 0.97 MNIST test accuracy)

Write a CNN that is strong enough to clear 0.97 in 3 epochs.

**Spec**

* Two `nn.Conv2d` blocks with `padding=1` and `nn.MaxPool2d(2)` between them.
* Two channel widths that roughly double: e.g. 32 → 64.
* A dense head: Flatten → Linear → ReLU → Linear(10).
* Train: 3 epochs, Adam, lr=1e-3.

Grading cell calls `check_q2(final_test_accuracy)`. Threshold: **≥ 0.97**.


In [ ]:
# Q2 — write a CNN here
#
# Spec:
#   - Two Conv2d blocks with padding=1 and MaxPool2d(2) between them.
#   - Channel widths roughly double: 32 → 64.
#   - Head: Flatten → Linear → ReLU → Linear(10).
#
# TODO: complete Q2CNN below.

class Q2CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=____),  # TODO: padding that preserves spatial size
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, ____, kernel_size=3, padding=1),  # TODO: second conv output channels
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(____, 128), nn.ReLU(),   # TODO: final conv feature-map size × channels
            nn.Linear(128, ____),              # TODO: output logits
        )

    def forward(self, x):
        return self.head(self.conv(x))


In [ ]:
# Q2 — train + grade (DO NOT EDIT)
torch.manual_seed(0)
q2_train, q2_test = load_mnist()
q2_acc = train_classifier(Q2CNN(), q2_train, q2_test, DEVICE, epochs=3, lr=1e-3, label='q2')
q2_pass, q2_score, q2_msg = check_q2(q2_acc)
print('\n' + q2_msg)


---

## Q3 — Build an LSTM for a Sequential Signal (10 min, ≥ 0.85 halves-task accuracy)

The `make_q3_dataset()` helper produces a synthetic binary-sequence task: for each
length-20 Bernoulli sequence, the label is 1 iff more 1s are in the **first half**.
A bag-of-words MLP scores ~0.50; an LSTM that correctly reads the last hidden
state clears 0.85+.

**Spec**

* `nn.LSTM(input_size=1, hidden_size=32, batch_first=True)`
* `nn.Linear(32, 2)` head that classifies on the FINAL timestep hidden state.
* Train: 5 epochs, Adam, lr=1e-3.

Grading cell calls `check_q3(final_test_accuracy)`. Threshold: **≥ 0.85**.


In [ ]:
# Q3 — write an LSTM here
#
# Spec:
#   - nn.LSTM(input_size=1, hidden_size=32, batch_first=True)
#   - nn.Linear(32, 2) head that classifies on the FINAL timestep.
#
# TODO: complete Q3LSTM below. The key is reading the last hidden
#       state from the LSTM's output tensor, not the full sequence.

class Q3LSTM(nn.Module):
    def __init__(self, hidden: int = 32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=____, hidden_size=____, batch_first=True)  # TODO
        self.head = nn.Linear(____, 2)                                              # TODO

    def forward(self, x):
        h, _ = self.lstm(x)                 # h shape: (batch, time, hidden)
        return self.head(h[____, ____, :])  # TODO: classify on the LAST timestep


In [ ]:
# Q3 — train + grade (DO NOT EDIT)
torch.manual_seed(0)
q3_train, q3_test = make_q3_dataset()
q3_acc = train_classifier(Q3LSTM(), q3_train, q3_test, DEVICE, epochs=5, lr=1e-3, label='q3')
q3_pass, q3_score, q3_msg = check_q3(q3_acc)
print('\n' + q3_msg)


---

## Q4 — Diagnose a Broken CNN (10 min, free-text answer)

We ship you a pre-built CNN (`Q4BrokenCNN`). The diagnostics report below
will surface an unhealthy gradient-flow pattern. Your job is to **diagnose**
the architectural bug and **prescribe** the fix in one short sentence.

Hints: the CNN has two `Conv2d` blocks starting from 28×28 MNIST input.
Think about what happens to the **spatial dimensions** as data flows through.

Grading cell calls `check_q4(MY_ANSWER)`. The checker accepts any phrasing
that mentions BOTH the diagnosed cause (stride / padding / dim collapse)
AND the fix (add padding or reduce stride).


In [ ]:
# Q4 — pre-built model + pre-built diagnostic run (DO NOT EDIT)
torch.manual_seed(0)
q4_train, q4_test = load_mnist()
q4_findings, q4_acc = train_and_diagnose(
    Q4BrokenCNN(),
    train_loader=q4_train, test_loader=q4_test, device=DEVICE,
    lr=1e-3, weight_decay=0.0, warmup_steps=0, epochs=3,
)
print('\nQ4 Diagnostics findings:')
for key, verdict in q4_findings.items():
    print(f'  {key:14s}: {verdict["severity"]:9s} — {verdict["message"]}')
print(f'\nQ4 broken CNN final test_acc: {q4_acc:.4f}')


In [ ]:
# Q4 — your answer
# Read the diagnostics output above and describe in one sentence:
#   (1) the architectural bug, and (2) the concrete fix.
#
# Your answer will be scored on keyword match — mention BOTH:
#   * what went wrong (stride / padding / dimension collapse)
#   * how to fix it (add padding or reduce stride)

MY_ANSWER = "____"   # TODO: replace with your one-sentence diagnosis + fix

q4_pass, q4_msg = check_q4(MY_ANSWER)
print(q4_msg)


---

## Q5 — Train and Prescribe (30–45 min, iterate until PASS)

This is the capstone. Fashion-MNIST. Deep MLP. The harness does the
training AND the diagnostics; your job is to produce an architecture
and a hyperparameter configuration that clears all three diagnostic
gates plus the accuracy threshold.

**Passing criteria**

* `dead_neurons == HEALTHY` — no dead ReLUs.
* `loss_trend == HEALTHY` — loss converging, not oscillating.
* `test_acc >= 0.82` — Fashion-MNIST test accuracy after 3 epochs.
* `gradient_flow` is printed advisory-only (see slide 5F for why the
  library's 1e-2 RMS cut-off is biased by 10-way softmax output layers).

**Iteration loop**

1. Edit Cell 1 below (the model + hyperparameters).
2. Run Cell 2 (fixed scaffolding — DO NOT EDIT).
3. Read the Prescription Pad. If FAIL, find the unhealthy verdict and
   consult the deck slide it references (5C Stethoscope, 5F Blood Test,
   5H X-Ray).
4. Return to step 1 and iterate until PASS.

Re-running Cell 2 creates a NEW model and trains from scratch each time
— there is no hidden state between iterations.


In [ ]:
# Q5 Cell 1 — YOUR WORK GOES HERE (student starter).
#
# The architecture below will FAIL `check_q5_pass`. Your job is to
# iteratively fix the model AND the hyperparameters until all three
# diagnostics verdicts line up with the passing criteria:
#
#     * dead_neurons == HEALTHY        (slide 5H — X-Ray)
#     * loss_trend   == HEALTHY        (slide 5C — Stethoscope)
#     * test_acc     >= 0.82           (slide 5J — Final Grade)
#     * gradient_flow is advisory — it may remain CRITICAL due to the
#       10-way softmax output layer; the grader does not block on it.
#
# Edit this cell, then re-run the NEXT cell. Iterate.

class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Starter: plain sequential with multiple issues for you to spot.
        self.net = nn.Sequential(
            nn.Linear(784, 512),
            nn.ReLU(),          # ← consider GELU? Check the Blood Test (slide 5F)
            nn.Linear(512, 256),
            nn.ReLU(),          # ← is this layer dying? (slide 5H — X-Ray)
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 10),  # ← output layer — no activation after this
        )

    def forward(self, x):
        return self.net(x.view(x.size(0), -1))

# Hyperparameters — the starter values will trip the Stethoscope (slide 5C).
LR             = 0.1     # ← too aggressive; re-read slide 5C on warmup
WEIGHT_DECAY   = 0.0     # ← no regularisation; see slide 5F
WARMUP_STEPS   = 0       # ← no warmup; loss will oscillate at step 0


In [ ]:
# Q5 Cell 2 — train + diagnose + grade (DO NOT EDIT)
torch.manual_seed(0)
q5_train, q5_val = load_fashion_mnist()
q5_findings, q5_test_acc = train_and_diagnose(
    MyModel(),
    train_loader=q5_train, test_loader=q5_val, device=DEVICE,
    lr=LR, weight_decay=WEIGHT_DECAY, warmup_steps=WARMUP_STEPS, epochs=3,
)
print('\n' + '═' * 68)
print('  Q5 Prescription Pad')
print('═' * 68)
for key in ('gradient_flow', 'dead_neurons', 'loss_trend'):
    v = q5_findings.get(key, {'severity': 'UNKNOWN', 'message': ''})
    print(f'  {key:14s} : {v["severity"]:9s} — {v["message"]}')
print(f'  test_acc       : {q5_test_acc:.4f}')
print('═' * 68)
q5_pass, q5_msg = check_q5_pass(q5_findings, q5_test_acc)
print('\n' + q5_msg)


**If FAIL**: iterate on Cell 1 and re-run Cell 2. Consult the deck slide that
names the prescription for each unhealthy verdict:

| Verdict                 | Slide | Prescription                                        |
| ----------------------- | ----- | --------------------------------------------------- |
| `loss_trend` unhealthy  | 5C    | Lower LR, add warmup, add LayerNorm.                |
| `gradient_flow` CRITICAL| 5F    | Lower LR, add weight decay, switch to GELU, clip.  |
| `dead_neurons` WARNING  | 5H    | Switch to GELU (or LeakyReLU), use Kaiming init.    |
| `test_acc` low          | 5J    | Re-check all three above; widen hidden dims.         |

The TARGET in this notebook clears PASS in ~20 seconds on T4. Your mileage
may vary ±1% due to CUDA non-determinism.

---


## Submission Summary

This final cell tallies all five questions. Total score is printed inline
and survives the notebook save so instructors see it when grading.


In [ ]:
# Submission summary (DO NOT EDIT)
scores = [
    ('Q1 MLP',      q1_pass,  f'{q1_score:.4f}'),
    ('Q2 CNN',      q2_pass,  f'{q2_score:.4f}'),
    ('Q3 LSTM',     q3_pass,  f'{q3_score:.4f}'),
    ('Q4 Diagnose', q4_pass,  'free-text'),
    ('Q5 Prescribe', q5_pass, f'test_acc={q5_test_acc:.4f}'),
]
total = sum(int(p) for _, p, _ in scores)
print('\n' + '═' * 48)
print('  MLFP05 Quiz — submission summary')
print('═' * 48)
for name, passed, score in scores:
    marker = '✓' if passed else '✗'
    status = 'PASS' if passed else 'FAIL'
    print(f'  {marker} {name:14s}: {status}  ({score})')
print('─' * 48)
print(f'  TOTAL: {total}/5')
print('═' * 48)
if total == 5:
    print('  Full pass. Push this notebook to your fork to submit.')
elif total == 4:
    print('  Conditional pass. Review the one FAIL with your instructor.')
else:
    print('  Not yet passing. Fix the FAILs and re-run the affected cells.')
